In [0]:
dbutils.widgets.removeAll()

In [0]:
tablas = [
    "catalog_au.silver.sfc_indicadores_v2",
    "catalog_au.golden.dim_entidad",
    "catalog_au.golden.dim_tiempo",
    "catalog_au.golden.dim_concepto_financiero"
]

for tabla in tablas:
    print(f"\n{'='*60}")
    print(tabla)
    print('='*60)
    spark.table(tabla).printSchema()

In [0]:
tablas = [
    "catalog_au.golden.dim_entidad",
    "catalog_au.golden.dim_tiempo",
    "catalog_au.golden.dim_concepto_financiero"
]

for tabla in tablas:
    print(f"\n{'='*60}")
    print(tabla)
    print('='*60)
    spark.table(tabla).printSchema()

In [0]:
from pyspark.sql import functions as F

df_silver = spark.table(
    "catalog_au.silver.sfc_indicadores_v2"
)

df_validos = df_silver.filter(
    (F.col("estado_validacion") == "VALIDO") &
    F.col("valor_indicador").isNotNull()
)

df_duplicados = (
    df_validos
    .groupBy(
        "codigo_entidad",
        "codigo_concepto",
        "fecha_corte"
    )
    .agg(F.count("*").alias("cantidad"))
    .filter(F.col("cantidad") > 1)
)

print("Registros Silver:", df_silver.count())
print("Registros válidos:", df_validos.count())
print("Combinaciones duplicadas:", df_duplicados.count())

display(df_duplicados.orderBy(F.desc("cantidad")).limit(20))

In [0]:
df_entidades = spark.table(
    "catalog_au.golden.dim_entidad"
)

df_conceptos = spark.table(
    "catalog_au.golden.dim_concepto_financiero"
)

df_tiempo = spark.table(
    "catalog_au.golden.dim_tiempo"
)

# Entidades sin correspondencia de código
sin_entidad = (
    df_validos
    .select("codigo_entidad")
    .distinct()
    .join(
        df_entidades.select("codigo_entidad").distinct(),
        "codigo_entidad",
        "left_anti"
    )
)

# Conceptos sin correspondencia
sin_concepto = (
    df_validos
    .select("codigo_concepto")
    .distinct()
    .join(
        df_conceptos.select("codigo_concepto").distinct(),
        "codigo_concepto",
        "left_anti"
    )
)

# Fechas sin correspondencia
sin_fecha = (
    df_validos
    .select("fecha_corte")
    .distinct()
    .join(
        df_tiempo.select("fecha").distinct(),
        F.col("fecha_corte") == F.col("fecha"),
        "left_anti"
    )
)

print("Códigos de entidad sin dimensión:", sin_entidad.count())
print("Códigos de concepto sin dimensión:", sin_concepto.count())
print("Fechas sin dimensión:", sin_fecha.count())

In [0]:
from pyspark.sql import functions as F

# Registros financieros válidos
df_fact_base = df_validos.alias("f")

# Dimensión entidad con historial SCD2
df_dim_entidad = df_entidades.alias("e")

# Relación por código y vigencia temporal
df_join_entidad = df_fact_base.join(
    df_dim_entidad,
    (
        (F.col("f.codigo_entidad") == F.col("e.codigo_entidad")) &
        (F.col("f.fecha_corte") >= F.col("e.fecha_inicio_vigencia")) &
        (
            F.col("e.fecha_fin_vigencia").isNull() |
            (F.col("f.fecha_corte") <= F.col("e.fecha_fin_vigencia"))
        )
    ),
    "left"
)

# Validar correspondencia SCD2 por registro de origen
df_control_scd2 = (
    df_join_entidad
    .groupBy(F.col("f.id_indicador").alias("id_indicador"))
    .agg(
        F.count("e.sk_entidad").alias("versiones_encontradas")
    )
)

display(
    df_control_scd2
    .groupBy("versiones_encontradas")
    .count()
    .orderBy("versiones_encontradas")
)

In [0]:
df_validos.agg(
    F.count("*").alias("total_registros"),
    F.countDistinct("id_indicador").alias("ids_unicos"),
    F.sum(
        F.when(F.col("id_indicador").isNull(), 1).otherwise(0)
    ).alias("ids_nulos")
).show()

In [0]:
from pyspark.sql import functions as F

df_fact_dimensional = (
    df_join_entidad
    .join(
        df_conceptos.alias("c"),
        F.col("f.codigo_concepto") == F.col("c.codigo_concepto"),
        "inner"
    )
    .join(
        df_tiempo.alias("t"),
        F.col("f.fecha_corte") == F.col("t.fecha"),
        "inner"
    )
    .select(
        F.col("f.id_indicador").alias("id_indicador"),
        F.col("e.sk_entidad").alias("sk_entidad"),
        F.col("c.sk_concepto").alias("sk_concepto"),
        F.col("t.sk_tiempo").alias("sk_tiempo"),
        F.col("f.valor_indicador").alias("valor_indicador"),
        F.col("f.unidad_medida").alias("unidad_medida"),
        F.col("f.fecha_corte").alias("fecha_corte"),
        F.col("f.archivo_origen").alias("archivo_origen"),
        F.col("f.fecha_ingesta").alias("fecha_ingesta"),
        F.current_timestamp().alias("fecha_procesamiento")
    )
)

print(
    "Registros dimensionales:",
    df_fact_dimensional.count()
)

display(df_fact_dimensional.limit(10))

In [0]:
df_fact_dimensional.agg(
    F.count("*").alias("total_registros"),
    F.countDistinct("id_indicador").alias("ids_unicos"),
    F.sum(
        F.when(F.col("sk_entidad").isNull(), 1).otherwise(0)
    ).alias("sk_entidad_nulas"),
    F.sum(
        F.when(F.col("sk_concepto").isNull(), 1).otherwise(0)
    ).alias("sk_concepto_nulas"),
    F.sum(
        F.when(F.col("sk_tiempo").isNull(), 1).otherwise(0)
    ).alias("sk_tiempo_nulas"),
    F.sum(
        F.when(F.col("valor_indicador").isNull(), 1).otherwise(0)
    ).alias("valores_nulos")
).show(truncate=False)

In [0]:
tabla_destino = (
    "catalog_au.silver.sfc_indicadores_fact_transform"
)

(
    df_fact_dimensional.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabla_destino)
)

print(f"Tabla Delta creada: {tabla_destino}")

In [0]:
from pyspark.sql import functions as F

df_persistido = spark.table(
    "catalog_au.silver.sfc_indicadores_fact_transform"
)

resultado = df_persistido.agg(
    F.count("*").alias("total_registros"),
    F.countDistinct("id_indicador").alias("ids_unicos"),
    F.countDistinct("sk_entidad").alias("entidades"),
    F.countDistinct("sk_concepto").alias("conceptos"),
    F.countDistinct("sk_tiempo").alias("periodos")
)

display(resultado)